# Reproducing Sargolini et al. (2006) from dandiset 000582

Paper: Sargolini F, Fyhn M, Hafting T, McNaughton BL, Witter MP, Moser M-B, Moser EI (2006).
Conjunctive representation of position, direction, and velocity in entorhinal cortex.
*Science* 312:758-762. [doi:10.1126/science.1125572](https://doi.org/10.1126/science.1125572)

Data: [DANDI:000582](https://dandiarchive.org/dandiset/000582/0.251111.2151), version
0.251111.2151, streamed from the archive. Nothing is downloaded to disk.

Why this version. The paper cites no version of the dandiset. 0.251111.2151 is the only published
version, and the draft (last modified 2026-05-28) holds the same 118 assets, with the same sizes
and asset identifiers, so there is no newer content to test. The earlier attempt,
`000582-sargolini-2006-01`, used the same version.

Claims. The claims file is `claims/000582/10.1126-science.1125572.yaml`, written by a reader from
the version of record and its Supporting Online Material (SOM), and checked by a second reader. It
replaces a file written from the abstract only. The three claims of the earlier attempt keep their
ids (`layer2-grid`, `deep-hd-conjunctive`, `speed`) and now carry the paper's numbers and criteria.
This notebook tests the claims file as it stands; where it leaves a detail open, every reading it
names is run. This attempt asked five questions about the claims file (`questions/000582.yaml`);
a reader answered them on 2026-10-20, the claims file now carries the answers, and this notebook
follows the updated criteria.

| Claim | Tested here |
|---|---|
| `layer2-grid` | tested |
| `grid-all-layers` | tested |
| `gridness-range-similar` | tested |
| `grid-stable` | tested |
| `spacing-dorsoventral` | blocked: no distance from the postrhinal border in the dandiset |
| `fieldsize-dorsoventral` | blocked: no distance from the postrhinal border in the dandiset |
| `grid-phase-orientation` | tested |
| `hd-layers` | tested |
| `hd-with-grid-colocalized` | tested |
| `hd-tuning-breadth-layers` | tested |
| `hd-full-range` | tested |
| `hd-colocalized-distributed` | tested |
| `hd-stable-trials` | tested |
| `hd-rates` | tested |
| `deep-hd-conjunctive` | tested |
| `gridness-hd-continuous` | tested |
| `conj-tuning-similar` | tested |
| `cue-rotation-coherent` | blocked: no file marks a cue-card rotation |
| `speed` | tested |
| `speed-slope-steeper` | tested |
| `speed-intercept` | tested |

The measures follow the SOM as the claims file quotes it, and are defined in `sargolini.py`
next to this notebook, with the source of each parameter.

Readings. The claims file names readings for details the paper leaves open (O1 to O4): trials
pooled or the first trial (O1), 3 cm maps with peaks of 25 bins or 1.5 cm maps with peaks of 100
pixels (O2), the gridness circle to the outermost peak's edge or its centre, and the central peak
removed as its pixels or as a disc (O3), with a peak's centre its pixel of highest correlation or
its centroid and its radius that of a disc of the same area or the largest distance from its centre
to one of its pixels (O3, as answered), and a significance level of 0.05 or 0.01 for the
head-direction tests (O4). The peak radius is not used when the circle runs to the outer peak's
centre and the central peak is removed as its pixels, so that pair of readings is run once. That
gives 56 combinations for grid cells, 4 for head-direction cells and 112 for conjunctive cells;
every claim is shown under all of them.

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import hashlib
import warnings
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import sargolini as sg

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
LAYERS = ["II", "III", "V", "VI"]
DEEP = ["III", "V", "VI"]


def within(value, expected, tolerance):
    return bool(abs(value - expected) <= tolerance + 1e-12)


def outcome_of(outcomes):
    """The outcome all readings give, or methods_underspecified when they differ."""
    found = sorted(set(outcomes))
    return found[0] if len(found) == 1 else "methods_underspecified (" + ", ".join(found) + ")"

## Data check

Each file is one recording: the positions of one or two tracking LEDs and the spike times of the
cells recorded. The units table gives each cell's name (tetrode and cell), layer, electrode depth
and hemisphere. There is no trial table and no other annotation.

In [ ]:
sessions = sg.load_sessions()
units = pd.DataFrame([
    dict(subject=s.subject, path=s.path, session=s.session_id, unit=u.name, layer=u.layer or "not given",
         depth_mm=u.depth_mm, hemisphere=u.hemisphere or "not given", two_leds=s.green is not None,
         minutes=int(round(s.duration / 60)), spikes=len(u.spike_times),
         rate_hz=len(u.spike_times) / s.duration)
    for s in sessions for u in s.units])
print(f"{len(sessions)} files, {units.subject.nunique()} rats, {len(units)} cell-sessions\n")
print("Files and cells by number of LEDs and file length:")
print(units.groupby(["two_leds", "minutes"]).agg(files=("path", "nunique"), cells=("unit", "size"),
                                                 rats=("subject", "nunique")).to_string())
print("\nCells by rat and layer:")
print(pd.crosstab(units.subject, units.layer, margins=True).to_string())
dts = np.concatenate([np.diff(s.t) for s in sessions])
print(f"\nPosition sampling interval: {dts.min():.4f} to {dts.max():.4f} s (median {np.median(dts):.4f}); "
      f"file lengths {min(s.duration for s in sessions):.1f} to {max(s.duration for s in sessions):.1f} s")
red = np.concatenate([s.red for s in sessions])
print(f"LED1 positions: x {np.nanmin(red[:, 0]):.1f} to {np.nanmax(red[:, 0]):.1f} cm, "
      f"y {np.nanmin(red[:, 1]):.1f} to {np.nanmax(red[:, 1]):.1f} cm; "
      f"missing samples {np.isnan(red).any(axis=1).mean():.2%}")
first_spike = min(u.spike_times.min() for s in sessions for u in s.units if len(u.spike_times))
outside = sum(int(np.sum((u.spike_times < s.t[0]) | (u.spike_times > s.t[-1] + s.dt)))
              for s in sessions for u in s.units)
print(f"Spikes per cell {units.spikes.min()} to {units.spikes.max()} (median {units.spikes.median():.0f}); "
      f"mean rate {units.rate_hz.min():.2f} to {units.rate_hz.max():.2f} Hz; "
      f"first spike at {first_spike:.3f} s; spikes outside the tracked period: {outside}")
print(f"Electrode depth {units.depth_mm.min():.2f} to {units.depth_mm.max():.2f} mm "
      f"(missing for {units.depth_mm.isna().sum()} cells); hemisphere missing for "
      f"{(units.hemisphere == 'not given').sum()} cells; layer missing for {(units.layer == 'not given').sum()} cells")
print("\nDistinct session descriptions:", Counter(s.description for s in sessions).most_common())

### The enclosure and the scale of the positions

Every file says the rat ran in a 1 x 1 m enclosure. The SOM lists four enclosures: circles of 90
and 180 cm and squares of 100 and 150 cm. Below, for each file: the span of the tracked positions,
whether they are centred so that the minimum is exactly minus half the larger span (which would
mean they were rescaled), the distance between the two LEDs (7 cm in the SOM, a check of the
scale that does not depend on the enclosure), and the fraction of samples in the corners outside
the inscribed circle (near zero in a circular enclosure).

In [ ]:
rows = []
for s in sessions:
    span = np.nanmax(s.red, axis=0) - np.nanmin(s.red, axis=0)
    big = span.max()
    centred = np.isclose(np.nanmin(s.red[:, np.argmax(span)]), -big / 2, atol=0.01)
    centre = (np.nanmax(s.red, axis=0) + np.nanmin(s.red, axis=0)) / 2
    r = np.hypot(*(s.red - centre).T)
    sep = np.hypot(*(s.green - s.red).T) if s.green is not None else np.array([np.nan])
    rows.append(dict(path=s.path, subject=s.subject, minutes=int(round(s.duration / 60)),
                     span_x=span[0], span_y=span[1], larger_span=round(big, 1), centred=centred,
                     led_separation=np.nanmedian(sep),
                     outside_circle=np.nanmean(r > big / 2 + 1)))
extent = pd.DataFrame(rows)
print("Files by larger span of positions (cm) and file length:")
print(extent.groupby(["larger_span", "minutes"]).agg(
    files=("path", "size"), rats=("subject", "nunique"), all_centred=("centred", "all"),
    median_led_separation_cm=("led_separation", "median"),
    median_outside_circle=("outside_circle", "median"), max_outside_circle=("outside_circle", "max")).round(3).to_string())
sep = extent.led_separation.dropna()
print(f"\nLED separation, median per file: {sep.min():.1f} to {sep.max():.1f} cm (median {sep.median():.1f}); SOM: 7 cm")
print(f"Files whose positions span exactly 100, 150 or 200 cm on the larger axis: "
      f"{extent.larger_span.isin([100.0, 150.0, 200.0]).sum()} of {len(extent)}")
print("Files with fewer than 1% of samples outside the inscribed circle (a circular enclosure), by span:",
      extent[extent.outside_circle < 0.01].groupby("larger_span").size().to_dict(), "of",
      extent.groupby("larger_span").size().to_dict())

### Trials

The SOM says every rat ran two 10-minute trials, 10 minutes apart. 95 files last 10 minutes and
23 last 20 minutes. Below, the step between successive position samples at 600 s in the 20-minute
files, against the 99th percentile of all steps in the same file: a jump at 600 s marks two trials
joined end to end.

In [ ]:
rows = []
for s in sessions:
    if len(s.trials) == 2:
        ok = ~np.isnan(s.red).any(axis=1)
        before = s.red[ok & (s.t < sg.TRIAL_S - 0.1)][-1]
        after = s.red[ok & (s.t >= sg.TRIAL_S + 0.1)][0]
        steps = np.hypot(*(s.red[10:] - s.red[:-10]).T)          # displacements over 0.2 s
        first = (s.t >= s.trials[0][0]) & (s.t < sg.TRIAL_S)
        centre_1 = (np.nanmax(s.red[first], axis=0) + np.nanmin(s.red[first], axis=0)) / 2
        centre_2 = (np.nanmax(s.red[~first], axis=0) + np.nanmin(s.red[~first], axis=0)) / 2
        rows.append(dict(file=s.path.split("/")[1][:22], step_599_9_to_600_1_cm=np.hypot(*(after - before)),
                         p99_9_of_0_2s_steps_cm=np.nanpercentile(steps, 99.9),
                         trial1_centre_offset_cm=np.hypot(*centre_1), trial2_centre_offset_cm=np.hypot(*centre_2)))
joins = pd.DataFrame(rows)
print(joins.round(2).to_string())
print(f"\n{(joins.step_599_9_to_600_1_cm > joins.p99_9_of_0_2s_steps_cm).sum()} of {len(joins)} files move farther "
      f"from 599.9 to 600.1 s than in 99.9% of their 0.2-s steps; the first trial's positions are centred "
      f"on zero (within 0.1 cm) in {(joins.trial1_centre_offset_cm < 0.1).sum()}, the second's in "
      f"{(joins.trial2_centre_offset_cm < 0.1).sum()}.")
same_day = units.assign(day=units.session.str[:6]).groupby(["subject", "day"]).path.nunique()
print(f"Days with more than one file: {(same_day > 1).sum()}; the files of a day are listed below.")

### Recordings deposited more than once, and the identity of cells across files

Identical spike trains or identical position tracks in two files would be a recording deposited
twice. Cells are listed per file; no identifier links a cell across files. Cells of the same rat
with the same electrode depth, hemisphere and tetrode-cell name in several files may be the same
cell recorded on several days.

In [ ]:
spike_hash = Counter(hashlib.md5(np.round(u.spike_times, 6).tobytes()).hexdigest()
                     for s in sessions for u in s.units)
track_hash = Counter(hashlib.md5(np.nan_to_num(s.red).tobytes()).hexdigest() for s in sessions)
print(f"Spike trains deposited more than once: {sum(v > 1 for v in spike_hash.values())}; "
      f"position tracks deposited more than once: {sum(v > 1 for v in track_hash.values())}")
key = ["subject", "depth_mm", "hemisphere", "unit"]
repeats = units[units.depth_mm.notna()].groupby(key).path.nunique()
print(f"Cell-sessions with depth given: {units.depth_mm.notna().sum()}; distinct (rat, depth, hemisphere, name): "
      f"{len(repeats)}; of these, {(repeats > 1).sum()} appear in 2 to {repeats.max()} files")
same_day = units.assign(day=units.session.str[:6])
for (subject, day), g in same_day.groupby(["subject", "day"]):
    if g.path.nunique() > 1:
        print(f"  rat {subject}, day {day}: " + "; ".join(
            f"{p.split('_ses-')[1][:8]} depth {h.depth_mm.iloc[0]:.2f} mm, {len(h)} cells"
            for p, h in g.groupby("path")))

### Cells by layer against table S1

Table S1A gives the cells sorted per layer for all 17 rats, table S1B for the nine two-LED rats.

In [ ]:
S1A = {"II": 108, "III": 223, "V": 110, "VI": 186}     # table S1A, cells sampled (paper)
S1B = {"II": 64, "III": 175, "V": 74, "VI": 136}       # table S1B, two-LED rats (paper)
per_layer = pd.DataFrame({
    "dandiset, all": units.layer.value_counts(), "paper S1A": pd.Series(S1A),
    "dandiset, two LEDs": units[units.two_leds].layer.value_counts(), "paper S1B": pd.Series(S1B)}).fillna(0).astype(int)
per_layer.loc["total"] = per_layer.sum()
print(per_layer.to_string())
leds = units.groupby("subject").two_leds.agg(["min", "max"])
print(f"\nRats with two-LED files only: {int((leds['min']).sum())}; one LED only: {int((~leds['max']).sum())}; "
      f"both: {list(leds[leds['min'] != leds['max']].index)} (the SOM: nine rats had two LEDs, eight one)")
print(f"Rats: {units.subject.nunique()} (the paper: 17)")

### Measures for every cell

The next cell computes, for every cell, the rate maps, autocorrelograms and gridness, the
head-direction tuning and its tests, and the speed regressions, under every reading, and for every
pair of cells recorded together the offset of their rate-map crosscorrelogram (`sargolini.py`).

In [ ]:
with ProcessPoolExecutor(min(8, os.cpu_count() or 1)) as pool:
    results = list(pool.map(sg.session_measures, sessions, chunksize=1))
cells = pd.DataFrame([c for cs, _ in results for c in cs])
pairs = pd.DataFrame([p for _, ps in results for p in ps])
by_path = {s.path: s for s in sessions}
print(f"{len(cells)} cells and {len(pairs)} pairs of simultaneously recorded cells")
no_score = {g: int(cells[f"grid|{t}|{g}"].isna().sum()) for t in sg.TRIALS for g in sg.GRID_READINGS}
print(f"Cells without a gridness (no autocorrelogram peak besides the central one): "
      f"{min(no_score.values())} to {max(no_score.values())} of {len(cells)}, by reading")
two_led = cells[cells.two_leds]
print(f"Two-LED cells without a Watson U2 or block correlation: "
      f"{int(two_led['u2|pooled'].isna().sum())} and {int(two_led['block_r|pooled'].isna().sum())} of {len(two_led)}")

### Do spikes and positions share a clock?

There are no trials or events to align to, so the check shifts each cell's spike train against
the tracking by -2 to +2 s and recomputes the measures the claims rest on: Watson's U2 for the 30
two-LED cells with the highest U2, and the spatial information of the 3 cm rate map for the 30
cells with the highest gridness. If the clocks agree, both peak at a shift near zero; an offset
between them would move the peak.

In [ ]:
lags = np.round(np.arange(-2, 2.01, 0.25), 2)
top_hd = cells[cells.two_leds].nlargest(30, "u2|pooled")
top_grid = cells.nlargest(30, f"grid|pooled|{sg.GRID_READINGS[0]}")
unit_of = {(s.path, u.name): (s, u) for s in sessions for u in s.units}
u2_lag = np.array([sg.lagged_u2(*unit_of[(r.path, r.unit)], lags) for r in top_hd.itertuples()])
info_lag = np.array([sg.lagged_map_info(*unit_of[(r.path, r.unit)], lags) for r in top_grid.itertuples()])
clock = pd.DataFrame({"shift_s": lags,
                      "median U2 / U2 at 0": np.median(u2_lag / u2_lag[:, [list(lags).index(0)]], axis=0),
                      "median info / info at 0": np.median(info_lag / info_lag[:, [list(lags).index(0)]], axis=0)})
print(clock.round(3).to_string(index=False))
print(f"\nShift of the peak, per cell: U2 {Counter(lags[np.argmax(u2_lag, axis=1)]).most_common()}; "
      f"spatial information {Counter(lags[np.argmax(info_lag, axis=1)]).most_common()}")
print(f"Cells: {top_hd.subject.nunique()} rats and {top_hd.path.nunique()} files (U2); "
      f"{top_grid.subject.nunique()} rats and {top_grid.path.nunique()} files (information)")

### The findings of attempt `000582-sargolini-2006-01`, and new ones

The version is the same as the earlier attempt's, and the data are unchanged, so each finding is
re-checked on the same files with the outputs above.

| Finding of `-01` | Kind | Status | Shown by |
|---|---|---|---|
| Every file says the enclosure was 1 x 1 m, but in 23 sessions the positions span 1.5 to 2.5 m | metadata_error | Still present | Enclosure table: 2 files span 150 cm and 20 span 200 cm on the larger axis, 19 of the 20 circular (under 1% of samples outside the inscribed circle) |
| The descriptions say position is given for two LEDs, but 47 of 118 files have one | metadata_error | Still present | First table: 47 files, 167 cell-sessions with one LED; rat 11084 has files of both kinds |
| No identifier links a cell across files | identifier_missing | Still present | 584 cell-sessions with a depth form 425 distinct (rat, depth, hemisphere, name), 99 of them in 2 to 7 files |
| The dandiset is a sample whose selection is not described | selection_undocumented | Still present | 620 cell-sessions against the paper's 627 cells, 15 rats against 17, layer II 71 against 108 while layers III, V and VI hold slightly more than table S1A |
| The paper is not open access and the dandiset gives no analysis parameters | methods_unavailable | Partly fixed | This attempt read the methods in the SOM, supplied by a person; the paper is still not open access, the dandiset states no parameter, and no analysis code is public |

New findings:

- The positions are rescaled. In all 118 files the positions span exactly 100, 150 or 200 cm on
  the larger axis, and the first trial is centred on zero. Of the 20 files spanning 200 cm, 19
  look circular; the SOM's large circle is 180 cm across, so in those files distances and speeds
  are about 11% too large. The distance between the two LEDs (median 5.2 cm per file, 2.6 to 36 cm,
  against 7 cm in the SOM) is too variable to fix the scale. The files do not say that the
  positions were rescaled.
- The 23 twenty-minute files hold two 10-minute trials joined near 600 s: in all 23 the position
  jumps between 599.9 and 600.1 s by more than 99.9% of the file's 0.2-s steps, and only the first
  trial is centred. There is no trial table, and nothing says which 10-minute files, if any, are
  the two trials of one recording; the files of a day differ in depth or cells (list above).
- No file gives the recording sites' distance from the postrhinal border, which the paper's
  dorsoventral analyses use; `depth` is the depth the electrodes were lowered to.
- No file names its enclosure, a condition or the cue-card position, so the cue-rotation trials
  cannot be found.
- Six cells have no layer, 36 no depth and 230 no hemisphere.
- The descriptions disagree with the files: the experiment description says the sample holds
  "conjunctive cells and head direction cells from layers III and V", the dandiset description
  that it holds spike times "for recorded grid cells"; the files hold cells of layers II, III, V and
  VI and of every class.
- The files do not say which LED is the red one, whose positions the SOM uses. LED1 is the only LED
  in the single-LED files, and the SOM says single-LED rats carried a red LED, so LED1 is taken as
  red.

Spikes and positions are consistent with a shared clock: Watson's U2 and the spatial information
peak at zero shift for 29 and 28 of 30 cells, and the 10 spikes outside the tracked period fall
after the last sample.

## Claim `layer2-grid`

> The proportion of cells with grid structure depended on layer. Most well-separated layer II
> cells were grid cells (66 of 108), against a smaller proportion in layers III (78 of 223), V (17
> of 110) and VI (42 of 186).

Criterion: all cells of all rats, by layer; a grid cell has gridness above 0, under readings O1 to
O3. Reproduced when the layer II fraction is above 0.5 and above that of each deeper layer, and
the four fractions are within tolerance (0.611 +/- 0.061, 0.350 +/- 0.035, 0.155 +/- 0.015,
0.226 +/- 0.023); partially reproduced when the order holds but a fraction is outside tolerance.
Each cell-session counts as a cell, as the paper's 627 cells are about as many as the dandiset's
620 cell-sessions (data check). The six cells with no layer are left out.

In [ ]:
EXPECTED_GRID = {"II": (0.611, 0.061), "III": (0.350, 0.035), "V": (0.155, 0.015), "VI": (0.226, 0.023)}
known = cells[cells.layer.isin(LAYERS)].copy()
n_layer = known.layer.value_counts()
rows = []
for T, G, _ in sg.readings(with_level=False):
    grid = sg.classify(known, T, G, 0.05).grid
    frac = grid.groupby(known.layer).mean()
    order = frac["II"] > 0.5 and all(frac["II"] > frac[l] for l in DEEP)
    tol = all(within(frac[l], *EXPECTED_GRID[l]) for l in LAYERS)
    rows.append(dict(trials=T, gridness=G, **{l: frac[l] for l in LAYERS},
                     **{f"n {l}": int(grid[known.layer == l].sum()) for l in LAYERS}, total=int(grid.sum()),
                     order=order, in_tolerance=[l for l in LAYERS if within(frac[l], *EXPECTED_GRID[l])],
                     outcome="reproduced" if order and tol else "partially_reproduced" if order else "not_reproduced"))
grid_table = pd.DataFrame(rows)
print("Cells per layer:", n_layer[LAYERS].to_dict())
print(grid_table.round(3).to_string())
print(f"\nRange over the {len(grid_table)} readings:", {l: f"{grid_table[l].min():.3f} to {grid_table[l].max():.3f}" for l in LAYERS})
print("Expected:", {l: f"{v - t:.3f} to {v + t:.3f}" for l, (v, t) in EXPECTED_GRID.items()})
print("Readings with each layer within tolerance:", {l: int(grid_table.in_tolerance.apply(lambda x: l in x).sum()) for l in LAYERS})
print("Outcomes:", grid_table.outcome.value_counts().to_dict(), "->", outcome_of(grid_table.outcome))

In [ ]:
READING = ("pooled", sg.GRID_READINGS[0], 0.05)     # the first reading, used for figures and per-rat tables
known["grid"] = sg.classify(known, READING[0], READING[1], 0.05).grid
per_rat = known.groupby(["subject", "layer"]).grid.agg(grid_cells="sum", cells="size")
per_rat["fraction"] = (per_rat.grid_cells / per_rat.cells).round(2)
print(f"Per rat, reading {READING[:2]}:")
print(per_rat.unstack("layer").to_string())
dedup = known[known.depth_mm.notna()].drop_duplicates(["subject", "depth_mm", "hemisphere", "unit"])
print("\nVariant, one cell-session per (rat, depth, hemisphere, name), first file kept:",
      {l: f"{int(g.grid.sum())}/{len(g)} = {g.grid.mean():.3f}" for l, g in dedup.groupby("layer")})
fig, ax = plt.subplots(figsize=(6, 3.2))
col = f"grid|{READING[0]}|{READING[1]}"
bins = np.linspace(-1.5, 1.8, 34)
for l in LAYERS:
    ax.hist(known[known.layer == l][col].dropna(), bins=bins, histtype="step", density=True, label=f"layer {l}")
ax.axvline(0, color="k", lw=1)
ax.set_xlabel("Gridness"); ax.set_ylabel("Density"); ax.legend(frameon=False, fontsize=8)
print(f"Gridness plotted: {known[col].min():.2f} to {known[col].max():.2f}; none outside the axes "
      f"({int(((known[col] < bins[0]) | (known[col] > bins[-1])).sum())} cells outside)")

**Result: partially reproduced.** Under all 56 readings layer II has the largest fraction of grid
cells, 0.592 to 0.718, above 0.5 and above layers III (0.529 to 0.604), V (0.297 to 0.415) and VI
(0.354 to 0.485), so the order holds. The layer II fraction is within its tolerance under 32 of
the 56 readings, but no deeper layer is within tolerance under any reading: each is well above the
paper's fraction. Gridness above 0 makes 277 to 329 cells grid cells, against the paper's 203 of
627. Per rat, the layer II fractions are 0.41 to 0.90 and the deeper ones 0 to 0.67; of the two
rats recorded in layers II and III, layer II is higher in 10884 (0.60 against 0.43) and lower in
11265 (0.62 against 0.67). Counting each (rat, depth, hemisphere, name) once gives the same order
(0.680, 0.511, 0.356, 0.329).

The answer on a peak's centre and radius adds the radius as the largest distance to a pixel of the
peak. The outcome is the same under every reading, as before; the ranges widen a little (layer II
0.620 to 0.718 before).

The earlier attempt also called this claim partially reproduced, for another reason: it used a
grid score of its own with a shuffled-data threshold, under which a third of layer II cells were
grid cells. With the paper's criterion, gridness above 0, most layer II cells are grid cells, and
the claim falls short on the deeper layers' fractions.

## Claim `grid-all-layers`

> Grid cells with tessellating firing fields were found in all four principal cell layers of MEC
> (II, III, V and VI).

Criterion: grid cells as in `layer2-grid`; reproduced when at least one grid cell is found in each
layer.

In [ ]:
counts = grid_table[[f"n {l}" for l in LAYERS]]
print(f"Fewest grid cells per layer over the {len(counts)} readings:", counts.min().to_dict())
print("Rats contributing grid cells per layer (first reading):",
      known[known.grid].groupby("layer").subject.nunique().to_dict())
all_layers = (counts > 0).all(axis=1)
print(f"Readings with a grid cell in every layer: {int(all_layers.sum())} of {len(all_layers)}")

**Result: reproduced.** Every layer has grid cells under all 56 readings: at least 42 in layer
II, 120 in III, 35 in V and 70 in VI, from 5, 6, 3 and 3 rats under the first reading.

## Claim `gridness-range-similar`

> Among deeper-layer cells with periodic firing, the range of gridness did not differ from that of
> layer II cells.

Criterion: a claim of absence with no test named by the paper. Grid cells of each layer (readings
O1 to O3). Readings of the test: (a) two-sided Mann-Whitney, layer II against layers III to VI
pooled; (b) Kruskal-Wallis across the four layers; each at p < 0.05. Reproduced when no reading
finds a difference, not reproduced when all do, methods_underspecified when they disagree.

In [ ]:
rows = []
for T, G, _ in sg.readings(with_level=False):
    col = f"grid|{T}|{G}"
    g = known[sg.classify(known, T, G, 0.05).grid]
    by = {l: g[g.layer == l][col] for l in LAYERS}
    mw = stats.mannwhitneyu(by["II"], g[g.layer != "II"][col], alternative="two-sided").pvalue
    kw = stats.kruskal(*by.values()).pvalue
    rows.append(dict(trials=T, gridness=G, mann_whitney_p=mw, kruskal_wallis_p=kw,
                     **{f"median {l}": by[l].median() for l in LAYERS},
                     **{f"range {l}": f"{by[l].min():.2f} to {by[l].max():.2f}" for l in LAYERS}))
range_table = pd.DataFrame(rows)
print(range_table.round(4).to_string())
tests = pd.concat([range_table.mann_whitney_p, range_table.kruskal_wallis_p])
print(f"\nTests finding a difference (p < 0.05): {int((tests < 0.05).sum())} of {len(tests)} "
      f"(Mann-Whitney {int((range_table.mann_whitney_p < 0.05).sum())} of {len(range_table)}, Kruskal-Wallis "
      f"{int((range_table.kruskal_wallis_p < 0.05).sum())} of {len(range_table)}); p from {tests.min():.2g} to {tests.max():.2g}")
print(f"Mann-Whitney p {range_table.mann_whitney_p.min():.3f} to {range_table.mann_whitney_p.max():.3f}; "
      f"Kruskal-Wallis p {range_table.kruskal_wallis_p.min():.4f} to {range_table.kruskal_wallis_p.max():.4f}")
print("Median gridness of grid cells, range over readings:",
      {l: f"{range_table[f'median {l}'].min():.2f} to {range_table[f'median {l}'].max():.2f}" for l in LAYERS})

**Result: blocked, methods_underspecified.** The two tests the criterion names disagree. The
Mann-Whitney test of layer II against the deeper layers finds a difference under 8 of the 56
readings (p 0.009 to 0.993), while the Kruskal-Wallis test across the four layers finds one under
53 of 56 (p 0.0001 to 0.159). The Kruskal-Wallis difference lies among the deeper layers: the median
gridness of grid cells is 0.42 to 0.78 in layer III against 0.17 to 0.47 in layer V, with layer II
at 0.23 to 0.58. The ranges themselves are alike: in every layer they run from about 0 to between
1.14 and 1.49. With the radius reading added the tests still disagree (2 of 32 and 32 of 32
before).

## Claim `grid-stable`

> All 203 grid cells had stable periodic firing patterns within and between trials.

Criterion: grid cells as in `layer2-grid`. Between trials, for cells recorded in two trials (the
20-minute files): (a) a positive map correlation with Pearson p < 0.05, or (b) gridness above 0
in each trial's map. Within a trial: the same tests on the maps of (a) the first and second halves,
(b) odd and even minutes, of each trial. Reproduced when every grid cell passes under every
reading; partially reproduced when at least 90% do; methods_underspecified when the readings
disagree. A cell passes a combination when it passes the within-trial reading in each of its
trials and, if it was recorded in two trials, the between-trial reading. Under O1 (b) only the
first trial is used within trials.

In [ ]:
def stability(frame, T, G):
    bname, rest = G.split("|", 1)
    g = frame[sg.classify(frame, T, G, 0.05).grid]
    within_tests = {}
    for split, test in product(("halves", "minutes"), ("correlation", "gridness")):
        ok = pd.Series(True, index=g.index)
        for k in ([0] if T == "first" else [0, 1]):
            has = g.trials > k
            if test == "correlation":
                passed = (g[f"within|{k}|{split}|{bname}|r"] > 0) & (g[f"within|{k}|{split}|{bname}|p"] < 0.05)
            else:
                passed = (g[f"within|{k}|{split}|{bname}|0|{rest}"] > 0) & (g[f"within|{k}|{split}|{bname}|1|{rest}"] > 0)
            ok &= passed | ~has
        within_tests[(split, test)] = ok
    two = g.trials == 2
    between_tests = {
        "correlation": (g[f"between|{bname}|r"] > 0) & (g[f"between|{bname}|p"] < 0.05),
        "gridness": (g[f"grid|first|{G}"] > 0) & (g[f"between|{bname}|trial2|{rest}"] > 0),
    }
    rows = []
    for (split, wt), w in within_tests.items():
        for bt, b in between_tests.items():
            stable = w & (b | ~two)
            rows.append(dict(trials=T, gridness=G, within=f"{split}, {wt}", between=bt, grid_cells=len(g),
                             two_trial_cells=int(two.sum()), within_pass=w.mean(),
                             between_pass=b[two].mean() if two.any() else np.nan,
                             stable=stable.mean()))
    return rows

stab = pd.DataFrame([r for T, G, _ in sg.readings(with_level=False) for r in stability(known, T, G)])
stab["outcome"] = np.select([stab.stable >= 1, stab.stable >= 0.9], ["reproduced", "partially_reproduced"], "not_reproduced")
print("Under the first gridness reading (all eight test combinations):")
print(stab[(stab.trials == "pooled") & (stab.gridness == sg.GRID_READINGS[0])].round(3).to_string())
summary = stab.groupby(["within", "between"]).agg(stable_min=("stable", "min"), stable_max=("stable", "max"),
                                                  within_min=("within_pass", "min"), within_max=("within_pass", "max"),
                                                  between_min=("between_pass", "min"), between_max=("between_pass", "max"))
print(f"\nRange over the {stab.groupby(['trials', 'gridness']).ngroups} grid readings:")
print(summary.round(3).to_string())
print(f"\nGrid cells: {stab.grid_cells.min()} to {stab.grid_cells.max()} (paper 203 +/- 20); in two-trial files "
      f"{stab.two_trial_cells.min()} to {stab.two_trial_cells.max()}")
print(f"Outcomes over {len(stab)} combinations:", stab.outcome.value_counts().to_dict(), "->", outcome_of(stab.outcome))

**Result: blocked, methods_underspecified.** No combination of readings has every grid cell
stable, and the readings split between the two lower outcomes: 84 of the 448 combinations give
partially reproduced (at least 90% stable) and 364 give not reproduced. With map correlations
within trials (odd and even minutes) and between trials, 90.2% to 96.3% of grid cells are stable;
with gridness above 0 required in both parts of each trial, 49% to 65% are. Between trials, 84% to
95% of the 71 to 85 grid cells in two-trial files have a significant positive map correlation, and
49% to 70% have gridness above 0 in both trials. The number of grid cells, 277 to 329, is outside
203 +/- 20 under every reading. With the radius reading added the split is the same as before
(51 partially reproduced and 205 not reproduced of 256).

## Claims `spacing-dorsoventral` and `fieldsize-dorsoventral`: blocked

Both need each recording site's distance from the postrhinal border, measured in the histology
(SOM, Histology; fig. S3). The units table has `depth` ("the depth of the inserted electrodes in
meters"), the depth the tetrodes were lowered to, which is not that distance, and no other column
or file gives it. Both claims are blocked with `data_missing`.

## Claim `grid-phase-orientation`

> In all layers the spatial phase of nearby grid cells was distributed (their vertices were
> offset), while grid orientation was consistent across simultaneously recorded cells.

Criterion: pairs of grid cells recorded together (same file and trial). Orientation, compared
modulo 60 degrees, is consistent when the within-file pairwise differences are smaller than those
between files of different rats, by a one-sided Mann-Whitney test at p < 0.05 (a standard choice
named in the claim). The pairs of different rats are taken (a) within the same layer, (b) from all
layers (readings, as answered). Phase is distributed when for more than half the pairs the peak of
the rate-map crosscorrelogram nearest the centre is displaced from the centre by (a) more than a
fifth of the grid spacing (the mean of the two cells'), (b) more than the 3 cm bin. Reproduced when
both parts hold under every reading, readings O1 to O3 included, in each layer with at least one
simultaneous pair. Pairs are of cells of the same layer. The criterion names no lower outcome; when
phase holds everywhere and orientation in some layers under every reading, the claim holds in part,
which the protocol calls partially reproduced.

In [ ]:
def phase_orientation(T, G):
    bname, _, _, loc, _ = G.split("|")
    g = known[sg.classify(known, T, G, 0.05).grid]
    info = g.set_index(["path", "unit"])[[f"orientation|{T}|{bname}|{loc}", f"spacing|{T}|{bname}|{loc}", "layer", "subject"]]
    info.columns = ["orientation", "spacing", "layer", "subject"]
    p = pairs.join(info.add_suffix("_a"), on=["path", "unit_a"], how="inner").join(info.add_suffix("_b"), on=["path", "unit_b"], how="inner")
    p = p[p.layer_a == p.layer_b]
    d = np.abs(p.orientation_a - p.orientation_b) % 60
    p = p.assign(d_orient=np.minimum(d, 60 - d), offset=p[f"offset|{T}|{bname}|{loc}"],
                 fifth=(p.spacing_a + p.spacing_b) / 2 / 5)
    rows = []
    for l in LAYERS:
        q = p[p.layer_a == l]
        if len(q) == 0:
            continue
        between = {}
        for pool, cl in (("layer", g[g.layer == l]), ("all layers", g)):
            o = cl[f"orientation|{T}|{bname}|{loc}"].to_numpy() % 60
            s = cl.subject.to_numpy()
            i, j = np.triu_indices(len(cl), 1)
            dd = np.abs(o[i] - o[j])[s[i] != s[j]]
            between[pool] = np.minimum(dd, 60 - dd)
        p_of = {pool: stats.mannwhitneyu(q.d_orient, b, alternative="less").pvalue if len(b) else np.nan
                for pool, b in between.items()}
        rows.append(dict(trials=T, gridness=G, layer=l, pairs=len(q), files=q.path.nunique(), rats=q.subject.nunique(),
                         median_within=q.d_orient.median(), median_between_rats=np.median(between["layer"]),
                         orientation_p=p_of["layer"], orientation_p_all_layers=p_of["all layers"],
                         displaced_fifth=(q.offset > q.fifth).mean(), displaced_3cm=(q.offset > 3).mean(),
                         no_peak=int(q.offset.isna().sum())))
    return rows

po = pd.DataFrame([r for T, G, _ in sg.readings(with_level=False) for r in phase_orientation(T, G)])
po["orientation_holds"] = po.orientation_p < 0.05
po["orientation_holds_all_layers"] = po.orientation_p_all_layers < 0.05
po["phase_holds"] = (po.displaced_fifth > 0.5) & (po.displaced_3cm > 0.5)
print("First gridness reading, by layer:")
print(po[(po.trials == "pooled") & (po.gridness == sg.GRID_READINGS[0])].round(3).to_string())
print(f"\nBy layer over the {po.groupby(['trials', 'gridness']).ngroups} readings:")
print(po.groupby("layer").agg(pairs_min=("pairs", "min"), pairs_max=("pairs", "max"), rats=("rats", "max"),
                              orientation_p_min=("orientation_p", "min"), orientation_p_max=("orientation_p", "max"),
                              p_all_layers_max=("orientation_p_all_layers", "max"),
                              orientation_holds=("orientation_holds", "mean"),
                              displaced_fifth_min=("displaced_fifth", "min"), displaced_fifth_max=("displaced_fifth", "max"),
                              displaced_3cm_min=("displaced_3cm", "min"), displaced_3cm_max=("displaced_3cm", "max"),
                              median_within=("median_within", "median"), median_between=("median_between_rats", "median")).round(3).to_string())
po["phase_fifth"] = po.displaced_fifth > 0.5
po["phase_3cm"] = po.displaced_3cm > 0.5
rows = []
for (T, G), x in po.groupby(["trials", "gridness"]):
    for pool, col in (("same layer", "orientation_holds"), ("all layers", "orientation_holds_all_layers")):
        for phase in ("phase_fifth", "phase_3cm"):
            rows.append(dict(trials=T, gridness=G, pool=pool, phase=phase, both_every_layer=bool((x[col] & x[phase]).all()),
                             phase_every_layer=bool(x[phase].all()),
                             orientation_layers=",".join(x[x[col]].layer), orientation_fails=",".join(x[~x[col]].layer)))
po_readings = pd.DataFrame(rows)
print(f"\nReadings (trials, gridness, pool, phase): {len(po_readings)}; both parts hold in every layer under "
      f"{int(po_readings.both_every_layer.sum())}; phase holds in every layer under {int(po_readings.phase_every_layer.sum())}")
print(po_readings.groupby(["pool", "phase"]).both_every_layer.agg(["sum", "size"]).to_string())
print("Layers where orientation fails, readings:", po_readings.orientation_fails.str.split(",").explode().replace("", np.nan).dropna().value_counts().to_dict())
for l in LAYERS:
    for pool, col in (("same layer", "orientation_holds"), ("all layers", "orientation_holds_all_layers")):
        x = po[po.layer == l]
        print(f"  layer {l}, other rats from {pool}: orientation holds under {int(x[col].sum())} of {len(x)} readings")
# Reproduced only when both parts hold in every layer under every reading (the criterion). Otherwise, partially
# reproduced when phase holds everywhere and orientation holds in some layer under every reading.
orientation_somewhere = po_readings.orientation_layers.ne("").all()
print("Outcome:", "reproduced" if po_readings.both_every_layer.all() else
      "partially_reproduced" if po_readings.phase_every_layer.all() and orientation_somewhere else "not_reproduced")


**Result: partially reproduced.** The phase part holds in every layer under every reading: 75% to
98.5% of simultaneous pairs have the crosscorrelogram peak nearest the centre displaced by more
than a fifth of the spacing, and 92.6% to 100% by more than 3 cm. Orientation is more alike within
files than between rats in layers II and III under every reading (p below 0.001), but not under
every reading in layers V (p up to 0.52; holds under 42 of 56 readings with the layer's own rats
and 41 with all layers) and VI (p up to 0.50; 35 and 23 of 56), which come from three rats each.
Both parts hold in every layer under 90 of the 224 readings (26 of 56 with the layer's own rats, 19
of 56 with all layers, the same under both phase readings). The criterion asks for every reading,
so the claim is not reproduced in full; it holds for phase in every layer and for orientation in
layers II and III.

Before the answer, the pool of other rats' pairs was asked, and so was whether "every reading"
covers O1 to O3. Read as a disagreement between readings, the claim was blocked as
methods_underspecified. The answer names both pools as readings and says "every reading" covers O1
to O3, so the claim needs both parts under all of them, and it is partially reproduced.

## Claim `hd-layers`

> No head-direction cells were found in layer II, whereas a large proportion of the cells in
> layers III to VI had directional preferences (273 of 385 cells, counting conjunctive cells).

Criterion: cells of two-LED files only; head-direction cells by the SOM criteria (Watson U2
significant and the odd/even-minute correlation significant) with readings O1 and O4; directional
cells are head-direction plus conjunctive cells. Reproduced when no layer II cell passes and the
fractions are within tolerance (deeper 0.709 +/- 0.071; III 0.669 +/- 0.067; V 0.878 +/- 0.088; VI
0.669 +/- 0.067); partially reproduced when no layer II cell passes and the deeper fraction is
above 0.5 but a number is outside tolerance. The dandiset holds the non-grid cells too (data
check), so the fractions can be formed.

In [ ]:
EXPECTED_HD = {"III to VI": (0.709, 0.071), "III": (0.669, 0.067), "V": (0.878, 0.088), "VI": (0.669, 0.067)}
tracked = known[known.two_leds].copy()
rows = []
for T, _, a in sg.readings(with_grid=False):
    hd = sg.classify(tracked, T, sg.GRID_READINGS[0], a).hd
    frac = {l: hd[tracked.layer == l].mean() for l in LAYERS}
    frac["III to VI"] = hd[tracked.layer != "II"].mean()
    n_ii = int(hd[tracked.layer == "II"].sum())
    tol = all(within(frac[k], *EXPECTED_HD[k]) for k in EXPECTED_HD)
    rows.append(dict(trials=T, level=a, layer_II=f"{n_ii}/{(tracked.layer == 'II').sum()}",
                     **{k: frac[k] for k in ["III to VI", "III", "V", "VI"]},
                     deeper=f"{int(hd[tracked.layer != 'II'].sum())}/{(tracked.layer != 'II').sum()}",
                     u2_significant_II=int((tracked[f"u2_p|{T}"] < a)[tracked.layer == "II"].sum()),
                     block_significant_II=int(((tracked[f"block_r|{T}"] > 0) & (tracked[f"block_p|{T}"] < a))[tracked.layer == "II"].sum()),
                     outcome="reproduced" if n_ii == 0 and tol else
                     "partially_reproduced" if n_ii == 0 and frac["III to VI"] > 0.5 else "not_reproduced"))
hd_table = pd.DataFrame(rows)
print("Two-LED cells per layer:", tracked.layer.value_counts()[LAYERS].to_dict())
print(hd_table.round(3).to_string())
print("Outcomes:", hd_table.outcome.value_counts().to_dict(), "->", outcome_of(hd_table.outcome))
for T, _, a in sg.readings(with_grid=False):
    hd = sg.classify(tracked, T, sg.GRID_READINGS[0], a).hd
    passing = tracked[hd & (tracked.layer == "II")]
    print(f"  layer II cells passing, {T}, {a}: rats {passing.subject.value_counts().to_dict()}, "
          f"files {sorted(passing.session.unique())}, U2 {passing[f'u2|{T}'].min():.2f} to {passing[f'u2|{T}'].max():.2f}")

In [ ]:
tracked["hd"] = sg.classify(tracked, READING[0], READING[1], READING[2]).hd
ii = tracked[(tracked.layer == "II")]
print(f"Layer II cells passing, reading {READING[0]}, {READING[2]}:")
print(ii[ii.hd][["subject", "session", "unit", "u2|pooled", "block_r|pooled", "block_p|pooled",
                 "grid|pooled|" + READING[1], "mv_r|pooled", "peak_rate|pooled"]].round(4).to_string())
print("\nPer rat and layer (two-LED cells):")
print(tracked.groupby(["subject", "layer"]).hd.agg(directional="sum", cells="size").unstack("layer").to_string())
print("\nLayer II, Watson U2 (median, range):", f"{ii['u2|pooled'].median():.1f}, {ii['u2|pooled'].min():.1f} to {ii['u2|pooled'].max():.1f};",
      "deeper head-direction cells:", f"{tracked[tracked.hd & (tracked.layer != 'II')]['u2|pooled'].median():.1f}")

**Result: not reproduced.** Under every reading some layer II cells pass both head-direction
criteria: 9, 7, 4 and 2 of 43. All of them are from rat 11265, in its two 20-minute files, and
their U2 (0.36 to 9.8) is below the median of the deeper head-direction cells (13.1). Watson's U2
alone is significant for 39 to 41 of the 43 layer II cells, so the block correlation decides which
pass. In the deeper layers 0.671 to 0.733 of cells are directional, within 0.709 +/- 0.071, and
layer V (0.924 to 0.937) is within its range; layer III (0.527 to 0.599) is below its range under
every reading and layer VI (0.713 to 0.790) above it under two of four. The claim needs no layer II
cell to pass even for a partial reproduction.

## Claim `hd-with-grid-colocalized`

> Beneath layer II, head-direction cells were usually recorded together with grid cells.

Criterion: two-LED files, layers III to VI, cell classes as in `hd-layers` and `layer2-grid`. The
unit of colocalization is (a) the file (one recording), (b) the tetrode within a file. The
head-direction cells are the pure head-direction cells of table S1B, and the cell found with them is
another cell of the unit: (a) a pure grid cell, (b) a grid or conjunctive cell (readings, as
answered). Reproduced when, under every reading, more than half of the units holding a
head-direction cell also hold such a cell; methods_underspecified when the readings disagree.

In [ ]:
deep = tracked[tracked.layer.isin(DEEP)]
rows = []
for T, G, a in sg.readings():
    c = sg.classify(deep, T, G, a)
    for companion, other in (("pure grid", c.grid & ~c.hd), ("grid or conjunctive", c.grid)):
        frame = deep.assign(pure_hd=c.pure_hd, other=other)
        for unit_name, keys in (("file", ["path"]), ("tetrode", ["path", "tetrode"])):
            g = frame.groupby(keys).agg(hd=("pure_hd", "any"), other=("other", "any"))
            with_hd = g[g.hd]
            rows.append(dict(trials=T, gridness=G, level=a, companion=companion, unit=unit_name, units_with_hd=len(with_hd),
                             also_grid=int(with_hd.other.sum()), fraction=with_hd.other.mean()))
coloc = pd.DataFrame(rows)
print(coloc.groupby(["companion", "unit"]).agg(units_min=("units_with_hd", "min"), units_max=("units_with_hd", "max"),
                                  fraction_min=("fraction", "min"), fraction_median=("fraction", "median"),
                                  fraction_max=("fraction", "max")).round(3).to_string())
print(coloc[(coloc.trials == READING[0]) & (coloc.gridness == READING[1]) & (coloc.level == READING[2])].round(3).to_string())
coloc["more_than_half"] = coloc.fraction > 0.5
print(f"\nReadings (trials, gridness, level, companion, unit) with more than half: {int(coloc.more_than_half.sum())} of {len(coloc)}")
print(coloc.groupby(["companion", "unit"]).more_than_half.agg(["sum", "size"]).to_string())
print("Outcome:", outcome_of(np.where(coloc.more_than_half, "reproduced", "not_reproduced")))

**Result: blocked, methods_underspecified.** Counted by file, 75% to 91% of the 49 to 57 files with a
pure head-direction cell in layers III to VI also hold a grid or conjunctive cell, and 47% to 60%
hold a pure grid cell (more than half under 83 of 112 readings). Counted by tetrode, 36% to 48% of
107 to 128 tetrodes hold a grid or conjunctive cell, and 14.5% to 19.4% a pure grid cell, below half
under every reading. The readings disagree. The answer confirmed the pure head-direction cells and
another cell, which this attempt had used, and added the pure grid cell as a reading; it lowers
the fractions but does not change the outcome.

## Claim `hd-tuning-breadth-layers`

> The breadth of directional tuning did not differ between layers III, V and VI (mean U2 18.2,
> 25.2 and 18.3; mean angular standard deviation 55.9, 52.2 and 58.2 degrees; P > 0.05).

Criterion: a claim of absence; tests (a) one-way ANOVA, (b) Kruskal-Wallis, each on U2 and on the
angular standard deviation; cells (a) pure head-direction cells, (b) with conjunctive cells. The
angular standard deviation is (a) Batschelet's angular deviation, sqrt(2(1 - r)), (b) his circular
standard deviation, sqrt(-2 ln r) (readings, as answered). Reproduced when no reading finds a
difference at p < 0.05; not reproduced when all do; methods_underspecified when they disagree; a
reproduced absence with means outside tolerance is partially reproduced.

In [ ]:
EXPECTED_U2 = {"III": (18.2, 1.82), "V": (25.2, 2.52), "VI": (18.3, 1.83)}
EXPECTED_ASD = {"III": (55.9, 5.59), "V": (52.2, 5.22), "VI": (58.2, 5.82)}
rows = []
for T, G, a in sg.readings():
    c = sg.classify(deep, T, G, a)
    for cell_reading, sel in (("pure head-direction", c.pure_hd), ("with conjunctive", c.hd)):
        if cell_reading == "with conjunctive" and G != sg.GRID_READINGS[0]:
            continue          # does not depend on the gridness reading
        g = deep[sel]
        for measure, col, exp in (("U2", f"u2|{T}", EXPECTED_U2), ("angular deviation", f"ang_dev|{T}", EXPECTED_ASD),
                                  ("circular sd", f"circ_sd|{T}", EXPECTED_ASD)):
            groups = [g[g.layer == l][col] for l in DEEP]
            means = {l: x.mean() for l, x in zip(DEEP, groups)}
            rows.append(dict(trials=T, gridness=G if cell_reading.startswith("pure") else "any", level=a, cells=cell_reading,
                             measure=measure, **{f"n {l}": len(x) for l, x in zip(DEEP, groups)},
                             **{f"mean {l}": means[l] for l in DEEP},
                             anova_p=stats.f_oneway(*groups).pvalue, kruskal_p=stats.kruskal(*groups).pvalue,
                             means_in_tolerance=all(within(means[l], *exp[l]) for l in DEEP)))
breadth = pd.DataFrame(rows)
print(breadth.groupby(["cells", "measure"]).agg(
    readings=("anova_p", "size"), anova_p_min=("anova_p", "min"), anova_p_max=("anova_p", "max"),
    kruskal_p_min=("kruskal_p", "min"), kruskal_p_max=("kruskal_p", "max"),
    **{f"mean {l} min": (f"mean {l}", "min") for l in DEEP}, **{f"mean {l} max": (f"mean {l}", "max") for l in DEEP},
    in_tolerance=("means_in_tolerance", "sum")).round(3).T.to_string())
print("\nWith conjunctive cells, every trial and level reading:")
print(breadth[breadth.cells == "with conjunctive"].round(3).to_string())
ps = pd.concat([breadth.anova_p, breadth.kruskal_p])
for m in ("U2", "angular deviation", "circular sd"):
    sub = breadth[breadth.measure == m]
    print(f"{m}: ANOVA finds a difference in {int((sub.anova_p < 0.05).sum())} of {len(sub)}, Kruskal-Wallis in "
          f"{int((sub.kruskal_p < 0.05).sum())} of {len(sub)}; means within tolerance in {int(sub.means_in_tolerance.sum())}")
print(f"\nTests finding a difference: {int((ps < 0.05).sum())} of {len(ps)}; "
      f"U2 tests {int((pd.concat([breadth[breadth.measure == 'U2'].anova_p, breadth[breadth.measure == 'U2'].kruskal_p]) < 0.05).sum())} "
      f"of {2 * (breadth.measure == 'U2').sum()}")

**Result: blocked, methods_underspecified.** On U2, no ANOVA and 10 of 116 Kruskal-Wallis tests
find a difference between layers III, V and VI; on the angular standard deviation many do, under
either formula (angular deviation: ANOVA 35 and Kruskal-Wallis 43 of 116; circular standard
deviation: 61 and 43 of 116). Layer V cells are the more sharply tuned: with conjunctive cells,
their angular deviation is 46.3 to 46.6 degrees against 52.6 to 55.0 in layers III and VI. The means
are within tolerance in 11 of 348 rows. The answer made both formulas readings, which this attempt
already ran; the readings still disagree, so the outcome is unchanged.

## Claim `hd-full-range`

> The entire range of head directions was represented by the preferred directions of
> head-direction cells in each of layers III, V and VI.

Criterion: head-direction and conjunctive cells of each layer; preferred direction is the bin of
peak rate. Reproduced when each (a) 30-degree, (b) 60-degree, (c) 90-degree sector holds the
preferred direction of at least one cell of the layer, in all three layers; not reproduced when no
reading holds in some layer; otherwise methods_underspecified. Sectors start at 0 degrees of the
camera's frame (a standard choice).

In [ ]:
rows = []
for T, _, a in sg.readings(with_grid=False):
    hd = sg.classify(deep, T, sg.GRID_READINGS[0], a).hd
    for l in DEEP:
        pref = np.degrees(deep[hd & (deep.layer == l)][f"peak_dir|{T}"]) % 360
        row = dict(trials=T, level=a, layer=l, cells=len(pref))
        for width in (30, 60, 90):
            filled = np.unique((pref // width).astype(int))
            row[f"{width} deg sectors filled"] = f"{len(filled)}/{360 // width}"
            row[f"{width} ok"] = len(filled) == 360 // width
        rows.append(row)
cover = pd.DataFrame(rows)
print(cover.to_string())
fig, axes = plt.subplots(1, 3, figsize=(9, 2.6), subplot_kw=dict(projection="polar"))
for ax, l in zip(axes, DEEP):
    pref = deep[tracked.loc[deep.index, "hd"] & (deep.layer == l)][f"peak_dir|{READING[0]}"]
    ax.hist(pref, bins=np.linspace(-np.pi, np.pi, 13))
    ax.set_title(f"layer {l}, n = {len(pref)}", fontsize=9); ax.set_yticklabels([])
print("Polar histograms: preferred directions, 30-degree bins, first reading; every cell is within the axes.")

**Result: reproduced.** Under all four readings the preferred directions of head-direction and
conjunctive cells fill all twelve 30-degree sectors, and so all 60- and 90-degree sectors, in each
of layers III (96 to 109 cells), V (73 to 74) and VI (102 to 113).

## Claim `hd-colocalized-distributed`

> Preferred directions of simultaneously recorded head-direction cells in the same area were
> widely distributed; in most ensembles of five or more cells (8 of 12) they showed no significant
> clustering.

Criterion: every file with 5 or more simultaneously recorded head-direction cells; the directions
of the cells' mean vectors tested for clustering with Watson's one-sample U2 test of uniformity at
p < 0.05, against the critical value for the number of cells (Stephens' small-sample form, which
gives the paper's 0.177 to 0.182 for 5 to 15 cells). Cells (a) pure head-direction cells, (b) with
conjunctive cells. Reproduced when, under a reading, fewer than half of the ensembles show
clustering; methods_underspecified when the readings disagree. Expected 4 +/- 0.4 clustered of
12 +/- 1.2 ensembles.

In [ ]:
rows, ensembles = [], {}
for T, G, a in sg.readings():
    c = sg.classify(deep, T, G, a)
    for cell_reading, sel in (("pure head-direction", c.pure_hd), ("with conjunctive", c.hd)):
        if cell_reading == "with conjunctive" and G != sg.GRID_READINGS[0]:
            continue
        found = []
        for path, g in deep[sel].groupby("path"):
            if len(g) >= 5:
                u2, crit = sg.watson_u2_uniformity(g[f"mv_dir|{T}"].to_numpy())
                found.append(dict(path=path, layer="/".join(sorted(g.layer.unique())), cells=len(g), u2=u2, critical=crit,
                                  clustered=u2 > crit))
        found = pd.DataFrame(found, columns=["path", "layer", "cells", "u2", "critical", "clustered"])
        ensembles[(T, G if cell_reading.startswith("pure") else "any", a, cell_reading)] = found
        rows.append(dict(trials=T, gridness=G if cell_reading.startswith("pure") else "any", level=a, cells=cell_reading,
                         ensembles=len(found), clustered=int(found.clustered.sum()),
                         fewer_than_half=found.clustered.sum() < len(found) / 2 if len(found) else np.nan))
distributed = pd.DataFrame(rows)
print(distributed.groupby("cells").agg(readings=("ensembles", "size"), ensembles_min=("ensembles", "min"),
                                       ensembles_max=("ensembles", "max"), clustered_min=("clustered", "min"),
                                       clustered_max=("clustered", "max"),
                                       readings_fewer_than_half=("fewer_than_half", "sum")).to_string())
print("\nWith conjunctive cells:")
print(distributed[distributed.cells == "with conjunctive"].to_string())
print("Ensembles per rat, with conjunctive cells, first reading:",
      ensembles[(READING[0], "any", READING[2], "with conjunctive")].path.str[4:9].value_counts().to_dict())
print("\nEnsembles, first reading with conjunctive cells:")
print(ensembles[(READING[0], "any", READING[2], "with conjunctive")].round(3).to_string())

**Result: reproduced.** Under every reading fewer than half of the ensembles of five or more
simultaneous head-direction cells show clustering by Watson's U2 test: 0 to 3 of 9 to 16
ensembles of pure head-direction cells, and 2 to 4 of 26 to 28 with conjunctive cells. The counts
differ from the paper's (12 ensembles, 4 clustered); the clustered count is within 4 +/- 0.4 under
one reading with conjunctive cells (4 of 26). Most ensembles come from rats 11207 (18 of 28) and 11265 (6).

## Claim `hd-stable-trials`

> Directional tuning curves were always stable across the two trials (circular correlation
> r = 0.84, P < 0.001).

Criterion: head-direction and conjunctive cells recorded in both trials (the 20-minute two-LED
files). Readings: (a) across cells, the circular-circular correlation of the preferred direction
in trial 1 with that in trial 2 (Jammalamadaka's coefficient, a standard choice); (b) per cell, the
Pearson correlation of the two trials' 60-bin tuning curves, averaged over cells, its mean tested
against 0 by a one-sample t-test. Reproduced when, under a reading, r is within 0.84 +/- 0.084
with p < 0.001; methods_underspecified when one reading reproduces and another fails.

In [ ]:
two_trial = deep[deep.trials == 2]
rows = []
for T, _, a in sg.readings(with_grid=False):
    g = two_trial[sg.classify(two_trial, T, sg.GRID_READINGS[0], a).hd]
    silent = g["trial_peak_dir|0"].isna() | g["trial_peak_dir|1"].isna()
    g = g[~silent]
    rcc, pcc = sg.circular_correlation(g["trial_peak_dir|0"].to_numpy(float), g["trial_peak_dir|1"].to_numpy(float))
    per_cell = []
    for t0, t1 in zip(g["trial_tuning|0"], g["trial_tuning|1"]):
        ok = ~np.isnan(t0) & ~np.isnan(t1)
        per_cell.append(stats.pearsonr(t0[ok], t1[ok])[0])
    per_cell = np.array(per_cell)
    rows.append(dict(trials=T, level=a, cells=len(g), dropped_silent_in_a_trial=int(silent.sum()),
                     files=g.path.nunique(), rats=g.subject.nunique(),
                     circular_r=rcc, circular_p=pcc, mean_curve_r=per_cell.mean(),
                     curve_r_range=f"{per_cell.min():.2f} to {per_cell.max():.2f}",
                     t_test_p=stats.ttest_1samp(per_cell, 0).pvalue,
                     cells_with_curve_r_below_0_5=int((per_cell < 0.5).sum())))
stable_hd = pd.DataFrame(rows)
stable_hd["a_reproduces"] = stable_hd.circular_r.between(0.756, 0.924) & (stable_hd.circular_p < 0.001)
stable_hd["b_reproduces"] = stable_hd.mean_curve_r.between(0.756, 0.924) & (stable_hd.t_test_p < 0.001)
print(stable_hd.round(4).to_string())
diff = pd.Series(np.degrees(np.angle(np.exp(1j * (two_trial["trial_peak_dir|1"] - two_trial["trial_peak_dir|0"]).astype(float)))),
                 index=two_trial.index)
hd_two = sg.classify(two_trial, READING[0], READING[1], READING[2]).hd & diff.notna()
print(f"\nChange of preferred direction between trials, first reading: median |change| "
      f"{np.median(np.abs(diff[hd_two])):.0f} degrees; within 30 degrees: {(np.abs(diff[hd_two]) <= 30).mean():.2f}; "
      f"by rat: {two_trial[hd_two].assign(d=np.abs(diff[hd_two])).groupby('subject').d.median().round(0).to_dict()}")

**Result: reproduced.** For the 88 to 96 head-direction and conjunctive cells in the 19 or 20
two-trial files (3 rats), the circular correlation of the preferred direction between trials is
0.78 to 0.85 and the mean correlation of the two trials' tuning curves 0.77 to 0.81, all with p
below 0.001 and within 0.84 +/- 0.084, under both readings and all four classifications. The
statement's "always" holds less well: 7 to 14 of these cells have tuning curves correlated below
0.5 between trials. Two cells silent in one trial are left out under each reading.

## Claim `hd-rates`

> In head-direction cells, firing rose from a low background rate (typically below 0.5 Hz) to a
> high maximum rate (5 to 40 Hz) when the rat's head faced the cell's range of directions.

Criterion: head-direction cells of the two-LED files, (a) pure, (b) with conjunctive cells. Peak
rate: the highest bin of the unsmoothed 60-bin tuning curve. Background rate: (a) the lowest bin,
(b) the mean over the bins more than 90 degrees from the preferred direction; "typically" is the
median over cells. Reproduced when, under a reading, the median background rate is below 0.55 Hz
and at least 90% of cells have peak rates from 4.5 to 44 Hz; methods_underspecified when readings
disagree.

In [ ]:
rows = []
for T, G, a in sg.readings():
    c = sg.classify(deep, T, G, a)
    for cell_reading, sel in (("pure head-direction", c.pure_hd), ("with conjunctive", c.hd)):
        if cell_reading == "with conjunctive" and G != sg.GRID_READINGS[0]:
            continue
        g = deep[sel]
        peak_ok = g[f"peak_rate|{T}"].between(4.5, 44).mean()
        for bg in ("bg_min", "bg_far"):
            med = g[f"{bg}|{T}"].median()
            rows.append(dict(trials=T, gridness=G if cell_reading.startswith("pure") else "any", level=a, cells=cell_reading,
                             background="lowest bin" if bg == "bg_min" else "beyond 90 deg", n=len(g),
                             median_background=med, peak_in_range=peak_ok,
                             median_peak=g[f"peak_rate|{T}"].median(),
                             reproduces=(med < 0.55) and (peak_ok >= 0.9)))
rates = pd.DataFrame(rows)
print(rates.groupby(["cells", "background"]).agg(
    readings=("n", "size"), median_background_min=("median_background", "min"),
    median_background_max=("median_background", "max"), peak_in_range_min=("peak_in_range", "min"),
    peak_in_range_max=("peak_in_range", "max"), median_peak_min=("median_peak", "min"),
    median_peak_max=("median_peak", "max"), reproducing=("reproduces", "sum")).round(3).to_string())
g = deep[tracked.loc[deep.index, "hd"]]
print(f"\nFirst reading, with conjunctive cells: peak rates {g['peak_rate|pooled'].min():.1f} to "
      f"{g['peak_rate|pooled'].max():.1f} Hz; below 4.5 Hz: {(g['peak_rate|pooled'] < 4.5).mean():.2f}; "
      f"lowest-bin rate {g['bg_min|pooled'].min():.2f} to {g['bg_min|pooled'].max():.2f} Hz")

**Result: not reproduced.** The background rate is low as stated: its median over cells is 0 Hz
in the lowest bin and 0.32 to 0.50 Hz beyond 90 degrees of the preferred direction, below 0.55 Hz
under every reading. But only 65.5% to 71.2% of head-direction cells peak between 4.5 and 44 Hz, where
the criterion needs 90%; the median peak is 5.9 to 6.9 Hz, and 34% of cells (first reading, with
conjunctive cells) peak below 4.5 Hz.

## Claim `deep-hd-conjunctive`

> The proportion of grid cells with conjunctive head-direction tuning depended on layer (66% in
> layer III, 90% in layer V, 28% in layer VI, none in layer II); the difference was significant
> across all four layers and across layers III to VI.

Criterion: two-LED files; grid and conjunctive cells by the SOM, readings O1 to O4. Chi-square
readings: (a) conjunctive against non-conjunctive grid cells per layer; (b) conjunctive cells
against all other cells per layer. Reproduced when, under a reading, both tests are significant
(p < 0.001 for four layers, p < 0.05 for III to VI) and the four fractions are within tolerance
(0, 0.66 +/- 0.066, 0.90 +/- 0.09, 0.28 +/- 0.028); partially reproduced when no layer II cell is
conjunctive and the four-layer test is significant but a fraction or the three-layer test fails.

In [ ]:
EXPECTED_CONJ = {"II": (0.0, 0.0), "III": (0.66, 0.066), "V": (0.90, 0.09), "VI": (0.28, 0.028)}
rows = []
for T, G, a in sg.readings():
    c = sg.classify(tracked, T, G, a)
    grid_cells = tracked[c.grid]
    frac = c.conj[c.grid].groupby(grid_cells.layer).mean().reindex(LAYERS)
    n_conj = c.conj.groupby(tracked.layer).sum().reindex(LAYERS)
    n_grid = c.grid.groupby(tracked.layer).sum().reindex(LAYERS)
    tables = {"a": pd.crosstab(grid_cells.layer, c.conj[c.grid]).reindex(index=LAYERS, columns=[False, True], fill_value=0),
              "b": pd.crosstab(tracked.layer, c.conj).reindex(index=LAYERS, columns=[False, True], fill_value=0)}
    tol = all(within(frac[l], *EXPECTED_CONJ[l]) for l in LAYERS)
    for chi, table in tables.items():
        chi4, p4 = stats.chi2_contingency(table, correction=False)[:2]
        chi3, p3 = stats.chi2_contingency(table.loc[DEEP], correction=False)[:2]
        outcome = ("reproduced" if p4 < 0.001 and p3 < 0.05 and tol else
                   "partially_reproduced" if n_conj["II"] == 0 and p4 < 0.001 else "not_reproduced")
        rows.append(dict(trials=T, gridness=G, level=a, chi_square=chi,
                         **{l: f"{n_conj[l]}/{n_grid[l]} = {frac[l]:.2f}" for l in LAYERS},
                         chi2_4=chi4, p_4=p4, chi2_3=chi3, p_3=p3, outcome=outcome))
conj_table = pd.DataFrame(rows)
print(conj_table[(conj_table.gridness == READING[1])].round(4).to_string())
fr = {l: conj_table[l].str.split("= ").str[1].astype(float) for l in LAYERS}
print(f"\nConjunctive fraction of grid cells, range over the {len(conj_table) // 2} readings:", {l: f"{v.min():.2f} to {v.max():.2f}" for l, v in fr.items()})
n_ii = conj_table.II.str.split("/").str[0].astype(int)
print(f"Layer II conjunctive cells: {n_ii.min()} to {n_ii.max()}; readings with none: {int((n_ii == 0).sum() / 2)} of {len(conj_table) // 2}")
print("chi-square p, four layers:", f"{conj_table.p_4.min():.2g} to {conj_table.p_4.max():.2g};",
      "layers III to VI:", f"{conj_table.p_3.min():.2g} to {conj_table.p_3.max():.2g}")
print(f"Outcomes over {len(conj_table)} rows:", conj_table.outcome.value_counts().to_dict(), "->", outcome_of(conj_table.outcome))
ii_rats = Counter()
for T, G, a in sg.readings():
    c = sg.classify(tracked, T, G, a)
    ii_rats.update(tracked[c.conj & (tracked.layer == "II")].subject)
print("Rats of the layer II conjunctive cells, summed over readings:", dict(ii_rats))

In [ ]:
c = sg.classify(tracked, *READING)
s1b = pd.crosstab(tracked.layer, np.select([c.conj, c.grid, c.pure_hd], ["conjunctive", "grid", "head direction"], "other"))
s1b = s1b.reindex(index=LAYERS, columns=["grid", "head direction", "conjunctive", "other"], fill_value=0)
print("Table S1B, first reading (the paper: II 34/0/0/30, III 19/80/37/39, V 1/56/9/8, VI 18/84/7/27):")
print(s1b.to_string())
print("\nConjunctive / grid cells per rat and layer, first reading:")
print(tracked.assign(conj=c.conj, grid=c.grid).groupby(["subject", "layer"]).agg(conj=("conj", "sum"), grid=("grid", "sum")).unstack("layer").to_string())

**Result: not reproduced.** Layer II has conjunctive cells under every reading (1 to 8, all from
rat 11265), and the claim needs none even for a partial reproduction. The conjunctive fraction of
grid cells is 0.86 to 0.95 in layer V, within 0.90 +/- 0.09, but 0.46 to 0.59 in layer III (paper
0.66) and 0.53 to 0.74 in layer VI (paper 0.28). The four-layer chi-square has p from 1.9e-11 to
0.23 and the three-layer one from 1.6e-05 to 0.78, depending on the reading. The radius reading
added by the answer changes none of this.

The earlier attempt called this claim reproduced. Its claim was the abstract's sentence, that
grid, head-direction and conjunctive cells occur together in the deeper layers, which the data
still show (table S1B above). The claim now carries the paper's per-layer fractions and requires
no conjunctive cell in layer II, and the data meet neither.

## Claim `gridness-hd-continuous`

> Grid cells and head-direction cells formed overlapping populations; the joint distribution of
> gridness and directional tuning was continuous, with some grid cells directionally tuned and
> some head-direction cells having grid structure.

Criterion: two-LED cells, gridness against Watson U2 (log scale) as in Fig. 3E. Reproduced when
conjunctive cells exist and, under readings O1 to O4, the cells passing either criterion form no
separate clusters, read as no gap along either axis wider than a quarter of that axis's range
among them (a standard choice).

In [ ]:
rows = []
for T, G, a in sg.readings():
    c = sg.classify(tracked, T, G, a)
    either = tracked[c.grid | c.hd]
    row = dict(trials=T, gridness=G, level=a, conjunctive=int(c.conj.sum()), cells=len(either))
    for axis, values in (("gridness", either[f"grid|{T}|{G}"].dropna()), ("log10 U2", np.log10(either[f"u2|{T}"].dropna()))):
        v = np.sort(values.to_numpy())
        gap = np.diff(v).max() / (v[-1] - v[0])
        row[f"largest gap, {axis} (fraction of range)"] = gap
    rows.append(row)
cont = pd.DataFrame(rows)
cont["continuous"] = (cont.conjunctive > 0) & (cont.filter(like="largest gap").max(axis=1) <= 0.25)
print(cont.describe().round(3).to_string())
print(f"\nReadings with conjunctive cells and no gap above a quarter: {int(cont.continuous.sum())} of {len(cont)}")
fig, ax = plt.subplots(figsize=(5, 3.6))
col, u2 = f"grid|{READING[0]}|{READING[1]}", f"u2|{READING[0]}"
c = sg.classify(tracked, *READING)
for label, sel, style in (("conjunctive", c.conj, "o"), ("other", ~c.conj, ".")):
    ax.scatter(tracked[sel][col], tracked[sel][u2], s=10, marker=style, label=label)
ax.set_yscale("log"); ax.set_xlabel("Gridness"); ax.set_ylabel("Watson U2"); ax.legend(frameon=False, fontsize=8)
print(f"Plotted: gridness {tracked[col].min():.2f} to {tracked[col].max():.2f}, U2 {tracked[u2].min():.2g} to {tracked[u2].max():.1f}; "
      f"{int(tracked[col].isna().sum())} cells without a gridness are not shown")

**Result: reproduced.** Under all 112 readings there are conjunctive cells (98 to 141), and among
the 366 to 395 cells passing either criterion the largest gap is at most 0.209 of the range of
gridness and 0.126 of the range of log U2, below a quarter.

## Claim `conj-tuning-similar`

> Directional tuning of conjunctive cells did not differ from that of pure head-direction cells
> (mean U2 18.0 against 19.4, P > 0.05).

Criterion: a claim of absence; (a) two-sided t-test, (b) two-sided Mann-Whitney test of U2,
conjunctive against pure head-direction cells, at p < 0.05. Reproduced when no reading finds a
difference; not reproduced when all do; methods_underspecified when they disagree; a reproduced
absence with means outside tolerance (18.0 +/- 1.8, 19.4 +/- 1.94) is partially reproduced. All
two-LED cells are used, as the paper's 53 and 220 cells span all layers.

In [ ]:
rows = []
for T, G, a in sg.readings():
    c = sg.classify(tracked, T, G, a)
    x, y = tracked[c.conj][f"u2|{T}"], tracked[c.pure_hd][f"u2|{T}"]
    rows.append(dict(trials=T, gridness=G, level=a, conjunctive=len(x), pure_hd=len(y), mean_conj=x.mean(), mean_hd=y.mean(),
                     median_conj=x.median(), median_hd=y.median(), t_p=stats.ttest_ind(x, y).pvalue,
                     mw_p=stats.mannwhitneyu(x, y, alternative="two-sided").pvalue))
conj_u2 = pd.DataFrame(rows)
conj_u2["in_tolerance"] = conj_u2.mean_conj.between(16.2, 19.8) & conj_u2.mean_hd.between(17.46, 21.34)
print(conj_u2.describe().round(4).to_string())
print(f"\nt-tests finding a difference: {int((conj_u2.t_p < 0.05).sum())} of {len(conj_u2)}; Mann-Whitney: {int((conj_u2.mw_p < 0.05).sum())} of {len(conj_u2)}; "
      f"means within tolerance: {int(conj_u2.in_tolerance.sum())} of {len(conj_u2)}")
print(conj_u2[(conj_u2.gridness == READING[1])].round(4).to_string())

**Result: blocked, methods_underspecified.** The t-test finds a difference in U2 between
conjunctive and pure head-direction cells under 98 of 112 readings (mean U2 12.0 to 19.4 against
18.2 to 25.4), the Mann-Whitney test under 42 of 112 (medians 8.8 to 15.5 against 10.5 to 17.4). The
means are within tolerance under 3 of 112 readings.

## Claim `cue-rotation-coherent`: blocked

The claim needs the cue-rotation trials of five rats in the small circular box, and the position of
the cue card in each. No file names its enclosure, its condition or a cue rotation, and there is no
trial table; the description says only that the rat ran in a 1 x 1 m enclosure. The rotated
trials cannot be found, so the claim is blocked with `data_missing`.

## Claim `speed`

> Regression showed a consistent positive speed-rate relation in nearly all grid cells (141 of
> 150), head-direction cells (153 of 220) and conjunctive cells (45 of 53).

Criterion: per cell, firing rate in 1-s bins regressed on speed over the trial, all bins kept.
Counting: (a) slope positive with P < 0.001; (b) slope positive, the count then tested against
half by a one-sided binomial sign test at p < 0.001. Grid cells are the grid cells that are not
conjunctive, from all rats; head-direction (pure) and conjunctive cells from two-LED files. Speed
from (a) path length, (b) displacement over the bin; positions (a) smoothed, (b) raw; trials as
O1; significance from (a) the slope, (b) Pearson's r (the same test as the slope for a simple
regression), (c) Spearman's rank correlation. As answered, the counting and significance readings
are readings of the reported number, and for each combination of method details (speed measure,
positions, trials and cell classes) the best of them is taken: reproduced when under one of them
all three fractions are within tolerance (0.94 +/- 0.094, 0.695 +/- 0.070, 0.849 +/- 0.085);
partially reproduced when none does but under one the grid and conjunctive fractions exceed the
head-direction fraction and all exceed one half. The claim takes that outcome when every
combination of method details gives it; methods_underspecified when they disagree.

In [ ]:
EXPECTED_SPEED = {"grid_only": (0.94, 0.094), "pure_hd": (0.695, 0.070), "conj": (0.849, 0.085)}
rows = []
for T, G, a in sg.readings():
    c = sg.classify(known, T, G, a)
    for sr in sg.SPEED_READINGS:
        slope = known[f"slope|{T}|{sr}"]
        for count, sig in (("P < 0.001, slope", f"p_linear|{T}|{sr}"), ("P < 0.001, Spearman", f"p_spearman|{T}|{sr}"),
                           ("positive slope", None)):
            passed = (slope > 0) & ((known[sig] < 0.001) if sig else True)
            row = dict(trials=T, gridness=G, level=a, speed=sr, count=count)
            for cls in EXPECTED_SPEED:
                sel = c[cls]
                row[f"{cls} n"] = int(sel.sum())
                row[cls] = passed[sel].mean()
                if sig is None:
                    row[f"{cls} sign p"] = stats.binomtest(int(passed[sel].sum()), int(sel.sum()), 0.5, alternative="greater").pvalue
            tol = all(within(row[k], *v) for k, v in EXPECTED_SPEED.items())
            order = row["grid_only"] > row["pure_hd"] and row["conj"] > row["pure_hd"] and min(row[k] for k in EXPECTED_SPEED) > 0.5
            row["outcome"] = "reproduced" if tol else "partially_reproduced" if order else "not_reproduced"
            rows.append(row)
speed_table = pd.DataFrame(rows)
print(speed_table.groupby(["count", "speed"]).agg(
    grid_min=("grid_only", "min"), grid_max=("grid_only", "max"), hd_min=("pure_hd", "min"), hd_max=("pure_hd", "max"),
    conj_min=("conj", "min"), conj_max=("conj", "max"),
    reproduced=("outcome", lambda o: int((o == "reproduced").sum())),
    partial=("outcome", lambda o: int((o == "partially_reproduced").sum())),
    readings=("outcome", "size")).round(3).to_string())
print("\nCells per class over the readings:", {k: f"{speed_table[f'{k} n'].min()} to {speed_table[f'{k} n'].max()}" for k in EXPECTED_SPEED})
sign = speed_table[speed_table["count"] == "positive slope"]
print("Sign test p (positive slope against half), largest over readings:",
      {k: f"{sign[f'{k} sign p'].max():.2g}" for k in EXPECTED_SPEED})
print("Outcomes over all rows:", speed_table.outcome.value_counts().to_dict())
# The counting and significance readings are readings of the reported number: for each combination of
# method details (trials, gridness, level, speed measure and positions) the best of them is taken.
RANK = {"reproduced": 0, "partially_reproduced": 1, "not_reproduced": 2}
DETAILS = ["trials", "gridness", "level", "speed"]
best = (speed_table.assign(rank=speed_table.outcome.map(RANK)).sort_values("rank", kind="stable")
        .drop_duplicates(DETAILS).reset_index(drop=True))
print(f"\nBest reading of the number per combination of method details ({len(best)} combinations):",
      best.outcome.value_counts().to_dict(), "->", outcome_of(best.outcome))
worst = best[best.outcome == "not_reproduced"][DETAILS]
print("Combinations not reproduced under any reading of the number, every reading:")
print(speed_table.merge(worst, on=DETAILS)[DETAILS + ["count", "grid_only", "pure_hd", "conj", "outcome"]].round(3).to_string())
print(best.groupby(["speed", "trials"]).outcome.value_counts().unstack(fill_value=0).to_string())
print(best.groupby("level").outcome.value_counts().unstack(fill_value=0).to_string())
print(best.assign(bin=best.gridness.str.split("|").str[0]).groupby("bin").outcome.value_counts().unstack(fill_value=0).to_string())
first = speed_table[(speed_table.trials == READING[0]) & (speed_table.gridness == READING[1]) & (speed_table.level == READING[2])]
print("\nFirst classification reading, every speed and count reading:")
print(first[["speed", "count", "grid_only n", "grid_only", "pure_hd n", "pure_hd", "conj n", "conj", "outcome"]].round(3).to_string())

In [ ]:
c = sg.classify(known, *READING)
cls = pd.Series(np.select([c.conj, c.grid_only, c.pure_hd], ["conjunctive", "grid", "head direction"], ""), index=known.index)
col = f"slope|{READING[0]}|path|smoothed"
sp = known.assign(cls=cls, positive_p=(known[col] > 0) & (known[f"p_linear|{READING[0]}|path|smoothed"] < 0.001))
print("Per rat, first reading, path length from smoothed positions, P < 0.001 and positive:")
print(sp[sp.cls != ""].groupby(["subject", "cls"]).positive_p.agg(["sum", "size"]).unstack("cls").to_string())
print(f"\nCorrelation of rate with speed per cell (median by class): "
      f"{sp[sp.cls != ''].groupby('cls')[f'r|{READING[0]}|path|smoothed'].median().round(3).to_dict()}")
fig, ax = plt.subplots(figsize=(5, 3))
for k in ["grid", "head direction", "conjunctive"]:
    ax.hist(sp[sp.cls == k][col], bins=np.linspace(-0.2, 0.5, 36), histtype="step", label=k, density=True)
ax.axvline(0, color="k", lw=1); ax.set_xlabel("Slope (Hz per cm/s)"); ax.legend(frameon=False, fontsize=8)
out = sp[sp.cls != ""][col]
print(f"Slopes {out.min():.3f} to {out.max():.3f} Hz per cm/s; outside the axes: {int(((out < -0.2) | (out > 0.5)).sum())}")

**Result: blocked, methods_underspecified.** Counted as cells with a positive slope at P < 0.001,
0.129 to 0.446 of grid cells, 0.094 to 0.282 of head-direction cells and 0.081 to 0.415 of
conjunctive cells pass, against 0.94, 0.695 and 0.849, under every combination of readings. Counted
by a positive slope, 0.815 to 0.886 of grid cells, 0.622 to 0.708 of head-direction cells and 0.686
to 0.822 of conjunctive cells pass. Taking the best reading of the number for each of the 448 combinations of method details,
119 give reproduced (all three fractions within tolerance), 327 partially reproduced (the order
holds) and 2 not reproduced (conjunctive 0.700 below head-direction 0.708, with displacement
speeds). The combinations disagree, so the claim is methods_underspecified. Reproduced
combinations are most frequent with path-length speeds and the first trial (37 and 33 of 56) and
rare with displacement speeds (2 to 11 of 56). Under the positive-slope count, the sign test
against half has p at most 5.5e-17, 1.5e-03 and 4.5e-05 for the three classes, so the
head-direction count misses p < 0.001 under some readings.

The question asked how to read the criterion, which called the claim reproduced "under a
combination of readings" and also methods_underspecified "when readings disagree". The answer
separates readings of the number, of which the best counts, from method details, which must agree.
The outcome is the same as before the answer, now for a stated reason: the method details
disagree.

The earlier attempt called this claim reproduced from a different measure (the correlation of rate
with speed over 5 cm/s bins, against shuffled spike trains) and a different claim (the abstract's
"all cell types were modulated by running speed").

## Claim `speed-slope-steeper`

> The slope of the speed-rate regression was steeper for conjunctive cells and grid cells than
> for head-direction cells, and did not differ between conjunctive and grid cells.

Criterion: slopes from the speed claim's regressions, its readings; one-way ANOVA and Tukey HSD.
Reproduced when the ANOVA has p < 0.005, Tukey gives conjunctive and grid each steeper than
head-direction at p < 0.05, and conjunctive against grid p > 0.05; partially reproduced when the
ANOVA is significant at p < 0.05 and one of the two comparisons with head-direction is.

In [ ]:
rows = []
for T, G, a in sg.readings():
    c = sg.classify(known, T, G, a)
    for sr in sg.SPEED_READINGS:
        col = f"slope|{T}|{sr}"
        g, h, j = known[c.grid_only][col], known[c.pure_hd][col], known[c.conj][col]
        anova = stats.f_oneway(g, h, j)
        tukey = stats.tukey_hsd(g, h, j).pvalue
        conj_hd = tukey[2, 1] < 0.05 and j.mean() > h.mean()
        grid_hd = tukey[0, 1] < 0.05 and g.mean() > h.mean()
        outcome = ("reproduced" if anova.pvalue < 0.005 and conj_hd and grid_hd and tukey[2, 0] > 0.05 else
                   "partially_reproduced" if anova.pvalue < 0.05 and (conj_hd or grid_hd) else "not_reproduced")
        rows.append(dict(trials=T, gridness=G, level=a, speed=sr, df=f"F(2,{len(g) + len(h) + len(j) - 3})", F=anova.statistic,
                         anova_p=anova.pvalue, mean_grid=g.mean(), mean_hd=h.mean(), mean_conj=j.mean(),
                         tukey_conj_hd=tukey[2, 1], tukey_grid_hd=tukey[0, 1], tukey_conj_grid=tukey[2, 0], outcome=outcome))
slopes = pd.DataFrame(rows)
print(slopes.groupby("speed").agg(F_min=("F", "min"), F_max=("F", "max"), anova_p_max=("anova_p", "max"),
                                  conj_hd_p_max=("tukey_conj_hd", "max"), grid_hd_p_max=("tukey_grid_hd", "max"),
                                  conj_grid_p_min=("tukey_conj_grid", "min"), conj_grid_p_max=("tukey_conj_grid", "max"),
                                  mean_grid=("mean_grid", "median"), mean_hd=("mean_hd", "median"), mean_conj=("mean_conj", "median")).round(4).to_string())
print(f"\nOver all readings: F from {slopes.F.min():.1f} to {slopes.F.max():.1f} with {sorted(slopes.df.unique())[0]} to "
      f"{sorted(slopes.df.unique())[-1]}; ANOVA p at most {slopes.anova_p.max():.3f}; Tukey p conjunctive against head-direction "
      f"{slopes.tukey_conj_hd.min():.2f} to {slopes.tukey_conj_hd.max():.2f}, grid against head-direction at most "
      f"{slopes.tukey_grid_hd.max():.4f}, conjunctive against grid {slopes.tukey_conj_grid.min():.4f} to {slopes.tukey_conj_grid.max():.2f}")
print("\nFirst classification reading:")
print(slopes[(slopes.trials == READING[0]) & (slopes.gridness == READING[1]) & (slopes.level == READING[2])].round(4).to_string())
print("\nOutcomes:", slopes.outcome.value_counts().to_dict(), "->", outcome_of(slopes.outcome))

**Result: blocked, methods_underspecified, because it depends on `speed`.** The analysis ran:
under all 448 combinations the ANOVA is significant (F 4.5 to 13.6, p at most 0.012) and grid
cells have steeper slopes than head-direction cells (Tukey p at most 0.0084), but conjunctive
cells do not differ from head-direction cells (Tukey p 0.06 to 0.99), which on its own would make
the claim partially reproduced under every reading. Conjunctive and grid cells differ under some
readings (Tukey p 0.0004 to 0.29). The degrees of freedom are F(2,444) to F(2,488).

## Claim `speed-intercept`

> The intercept of the speed-rate regression was positive in all cells but one (422 of 423), and
> mean rates at the lowest speeds (0 to 3 cm/s) were about 2.3 to 2.5 Hz in each cell class, so
> firing persisted during brief stops.

Criterion: regressions and readings as in the speed claim. Averages: (a) the mean over cells of
each cell's mean rate in the 1-s bins from 0 to 3 cm/s; (b) the mean over all such bins of the
class pooled. Reproduced when, under a reading, the fraction is within 0.998 +/- 0.0998 and the
rates within 2.26 +/- 0.03, 2.50 +/- 0.02 and 2.39 +/- 0.05 Hz; partially reproduced when the
fraction holds and the rates are within 10% of the printed values.

In [ ]:
EXPECTED_LOW = {"grid_only": 2.26, "pure_hd": 2.50, "conj": 2.39}
SPREAD_LOW = {"grid_only": 0.03, "pure_hd": 0.02, "conj": 0.05}
rows = []
for T, G, a in sg.readings():
    c = sg.classify(known, T, G, a)
    sel = c.grid_only | c.pure_hd | c.conj
    for sr in sg.SPEED_READINGS:
        positive = (known[f"intercept|{T}|{sr}"] > 0)[sel]
        for avg in ("mean of cells", "pooled bins"):
            row = dict(trials=T, gridness=G, level=a, speed=sr, average=avg, cells=int(sel.sum()),
                       positive_intercept=positive.mean(), negative=int((~positive).sum()))
            for k in EXPECTED_LOW:
                g = known[c[k]]
                if avg == "mean of cells":
                    row[k] = g[f"low_rate|{T}|{sr}"].mean()
                else:
                    row[k] = g[f"low_spikes|{T}|{sr}"].sum() / g[f"low_bins|{T}|{sr}"].sum()
            fraction_ok = within(row["positive_intercept"], 0.998, 0.0998)
            exact = all(within(row[k], EXPECTED_LOW[k], SPREAD_LOW[k]) for k in EXPECTED_LOW)
            ten = all(within(row[k], EXPECTED_LOW[k], 0.1 * EXPECTED_LOW[k]) for k in EXPECTED_LOW)
            row["outcome"] = ("reproduced" if fraction_ok and exact else
                              "partially_reproduced" if fraction_ok and ten else "not_reproduced")
            rows.append(row)
low = pd.DataFrame(rows)
print(low.groupby(["speed", "average"]).agg(
    positive_min=("positive_intercept", "min"), positive_max=("positive_intercept", "max"),
    negative_min=("negative", "min"), negative_max=("negative", "max"), grid_min=("grid_only", "min"), grid_max=("grid_only", "max"),
    hd_min=("pure_hd", "min"), hd_max=("pure_hd", "max"), conj_min=("conj", "min"), conj_max=("conj", "max"),
    reproduced=("outcome", lambda o: int((o == "reproduced").sum())),
    partial=("outcome", lambda o: int((o == "partially_reproduced").sum())), readings=("outcome", "size")).round(3).to_string())
lowbins = known[f"low_bins|{READING[0]}|path|smoothed"]
print(f"\n1-s bins at 0 to 3 cm/s per cell (first reading, path length, smoothed): median {lowbins.median():.0f}, "
      f"range {lowbins.min()} to {lowbins.max()}")
print("Outcomes:", low.outcome.value_counts().to_dict())

**Result: blocked, methods_underspecified, because it depends on `speed`.** The analysis ran: the
intercept is positive in 98.2% to 99.8% of cells (at most 8 negative), within tolerance, but the mean
rates at 0 to 3 cm/s are 1.2 to 1.9 Hz for grid cells, 1.1 to 1.8 Hz for head-direction cells and
0.9 to 1.6 Hz for conjunctive cells, more than 10% below the paper's 2.26, 2.50 and 2.39 Hz under
every reading, which on its own would make the claim not reproduced. Speeds depend on the scale
of the positions, which the files rescale (data check).

## Summary

| Claim | Outcome | Earlier attempt (`-01`) |
|---|---|---|
| `layer2-grid` | partially reproduced | partially reproduced |
| `grid-all-layers` | reproduced | not tested |
| `gridness-range-similar` | blocked: methods_underspecified | not tested |
| `grid-stable` | blocked: methods_underspecified | not tested |
| `spacing-dorsoventral` | blocked: data_missing | not tested |
| `fieldsize-dorsoventral` | blocked: data_missing | not tested |
| `grid-phase-orientation` | partially reproduced | not tested |
| `hd-layers` | not reproduced | not tested |
| `hd-with-grid-colocalized` | blocked: methods_underspecified | not tested |
| `hd-tuning-breadth-layers` | blocked: methods_underspecified | not tested |
| `hd-full-range` | reproduced | not tested |
| `hd-colocalized-distributed` | reproduced | not tested |
| `hd-stable-trials` | reproduced | not tested |
| `hd-rates` | not reproduced | not tested |
| `deep-hd-conjunctive` | not reproduced | reproduced |
| `gridness-hd-continuous` | reproduced | not tested |
| `conj-tuning-similar` | blocked: methods_underspecified | not tested |
| `cue-rotation-coherent` | blocked: data_missing | not tested |
| `speed` | blocked: methods_underspecified | reproduced |
| `speed-slope-steeper` | blocked: methods_underspecified (depends on `speed`) | not tested |
| `speed-intercept` | blocked: methods_underspecified (depends on `speed`) | not tested |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

The five questions this attempt asked were answered on 2026-10-20. One outcome changed:
`grid-phase-orientation`, blocked before, is partially reproduced, as the answer says every reading
must hold. The others keep their outcomes under the updated criteria.

How the analysis differs from the paper's:

- Each cell-session counts as a cell; the paper's count of 627 cells is about the dandiset's 620
  cell-sessions, but some cells may appear in several files.
- Trials: the 20-minute files are split at the jump in position near 600 s; the 10-minute files
  are taken as single trials.
- Positions are as deposited, rescaled (see the data check); LED1 is taken as the red LED.
- Details the claims file leaves open and that this attempt fixed as standard choices: contiguous
  means 4-connected; the block correlation
  passes when r > 0 with a two-sided p below the level; the small-sample critical value of
  Watson's one-sample U2 is Stephens'; the circular correlation is Jammalamadaka and SenGupta's;
  sectors for `hd-full-range` start at 0 degrees; a spike takes the position of the nearest sample.
- A peak's centre (its maximum or its centroid) and its radius (of a disc of the same area, or the
  largest distance to a pixel of the peak) are run as the readings the claims file now names; the
  central peak takes the same centre and radius.
- Under the paper's criterion, gridness above 0, this dandiset has more grid cells than the paper
  reports (277 to 329 against 203), and more in the deeper layers; this shifts every claim built on
  the grid class.

What this attempt found about the dandiset is listed in the data check: the five findings of the
earlier attempt are still present (one, the missing methods, partly fixed by the supplied paper),
and the new ones are the rescaled positions, the two trials joined in the 20-minute files with no
trial table, the missing distance from the postrhinal border, the missing enclosure and cue-card
labels, cells without layer, depth or hemisphere, descriptions that disagree with the files, and
no statement of which LED is red.